<a href="https://colab.research.google.com/github/Ronald-Tuncar/lab09/blob/develop/LAB_S09_Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 9: Redes Neuronales Artificiales


In [ ]:

# PARTE A - Clasificación de cáncer de mama (desde el dataset original de UCI)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Cargar datos desde el repositorio UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

# Asignar nombres a las columnas según la descripción del dataset
column_names = ['ID', 'Clump Thickness', 'Uniformity of Cell Size',
                'Uniformity of Cell Shape', 'Marginal Adhesion',
                'Single Epithelial Cell Size', 'Bare Nuclei',
                'Bland Chromatin', 'Normal Nucleoli', 'Mitoses', 'Class']

data = pd.read_csv(url, names=column_names)


In [ ]:
# Elimanamos los valores faltantes
data.replace('?', pd.NA, inplace=True)
data.dropna(inplace=True)

In [ ]:
# Convertimos a tipo numérico
data['Bare Nuclei'] = pd.to_numeric(data['Bare Nuclei'])

In [ ]:
# Separaramos características y etiqueta
X = data.drop(['ID', 'Class'], axis=1)
y = data['Class'].replace({2: 0, 4: 1})  # 2 = benigno (0), 4 = maligno (1)

In [ ]:
# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalamos los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Creamos y entrenamos el modelo
mlp = MLPClassifier(hidden_layer_sizes=(30, 30), max_iter=1000, activation='relu', solver='adam', random_state=42)
mlp.fit(X_train_scaled, y_train)

# Evaluación
y_pred = mlp.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9708029197080292
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        79
           1       0.97      0.97      0.97        58

    accuracy                           0.97       137
   macro avg       0.97      0.97      0.97       137
weighted avg       0.97      0.97      0.97       137



In [ ]:
# PARTE B - Clasificación de ropa con red neuronal convolucional (CNN)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical


In [ ]:
# Cargando el  dataset
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Preprocesamiento
train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Construcción del modelo CNN
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compilación y entrenamiento
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_images, train_labels, epochs=5, batch_size=64, validation_split=0.2)

# Evaluación
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test Accuracy: {test_acc}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 49s 62ms/step - accuracy: 0.7228 - loss: 0.7814 - val_accuracy: 0.8366 - val_loss: 0.4352
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 80s 59ms/step - accuracy: 0.8653 - loss: 0.3762 - val_accuracy: 0.8773 - val_loss: 0.3457
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 79s 56ms/step - accuracy: 0.8883 - loss: 0.3095 - val_accuracy: 0.8888 - val_loss: 0.3122
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 84s 59ms/step - accuracy: 0.8978 - loss: 0.2811 - val_accuracy: 0.8914 - val_loss: 0.3004
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 79s 55ms/step - accuracy: 0.9052 - loss: 0.2587 - val_accuracy: 0.8960 - val_loss: 0.2808
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8956 - loss: 0.2984
Test Accuracy: 0.8956000208854675
